# Random Forest Benchmark: scikit-learn, Bankai, and LightGBM

This notebook compares three workloads on the same synthetic binary dataset: **10,000 rows, 500 features (90 informative)**, a stratified 80/20 split, and a `float32` matrix. It retains the tree parameters from the previous notebook: 40 trees, maximum depth 20, up to 511 leaves, at least 5 samples per leaf, bootstrap sampling at 80%, all features, seed 42, and `n_jobs=-1`.

1. **Split importance:** scikit-learn RF with split counts extracted from its trees vs. exact Bankai (`max_bins=None`), without histogram acceleration.
2. **Permutation importance:** scikit-learn permutation importance on the validation set vs. native exact Bankai OOB importance.
3. **Histograms:** Bankai with 63 bins vs. LightGBM RF with 63 bins, both using gain importance.

Each variant has one warm-up run and ten measured runs in alternating order. Results are shown only in tables: individual measurements and per-model medians. Quality metrics (`accuracy`, `precision`, `recall`, `F1`, and `ROC-AUC`) and `predict_proba` calls are excluded from the timers. Bankai's native OOB permutation importance is computed inside `fit`; in scikit-learn it is an external step, so the tables also show fit time plus external importance time for that comparison.

The first cell installs the dependencies for Google Colab.

In [8]:
%pip install --upgrade -q "bankai-random-forest" "scikit-learn==1.9.1" "lightgbm==4.7.0"

/Users/jesus/Documents/Code/bankai-random-forest-v2/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


## 1. Runtime and dependencies

Check the available versions and CPUs before starting the measurements.

In [9]:
import gc
import os
import platform
import time
from importlib.metadata import version

import lightgbm as lgb
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from joblib import cpu_count
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from bankai_random_forest import BankaiRandomForestClassifier

print(f"Python: {platform.python_version()}")
print(f"Bankai: {version('bankai-random-forest')}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"LightGBM: {lgb.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Logical CPUs: {os.cpu_count()}; joblib CPUs: {cpu_count()}")

Python: 3.11.11
Bankai: 0.1.0a7
scikit-learn: 1.9.1
LightGBM: 4.7.0
NumPy: 2.4.6
Logical CPUs: 8; joblib CPUs: 8


## 2. Shared 10,000-row dataset

We generate the matrix once and share the same arrays across all models. The first 90 features contain signal; the remaining 410 are noise.

In [10]:
N_SAMPLES = 10_000
N_FEATURES = 500
N_RELEVANT = 90
DATA_SEED = 1789

rng = np.random.default_rng(DATA_SEED)
X = rng.standard_normal((N_SAMPLES, N_FEATURES), dtype=np.float32)
coefficients = rng.choice(np.array([-1.0, 1.0], dtype=np.float32), size=N_RELEVANT)
coefficients *= rng.uniform(0.5, 1.5, size=N_RELEVANT).astype(np.float32)
signal = X[:, :N_RELEVANT] @ coefficients
signal /= np.sqrt(np.sum(coefficients ** 2))
probability = 1.0 / (1.0 + np.exp(-signal))
y = rng.binomial(1, probability).astype(np.uint8)
X_train, X_validation, y_train, y_validation = train_test_split(
    X, y, test_size=0.2, random_state=DATA_SEED, stratify=y
)
assert X.shape == (10_000, 500) and X.dtype == np.float32
print(f"X={X.shape}, dtype={X.dtype}, {X.nbytes / 2**20:.1f} MiB")
print(f"Training={X_train.shape}; validation={X_validation.shape}; classes={np.bincount(y).tolist()}")
print(f"Informative features: 0–{N_RELEVANT - 1}; noise: {N_RELEVANT}–{N_FEATURES - 1}")

X=(10000, 500), dtype=float32, 19.1 MiB
Training=(8000, 500); validation=(2000, 500); classes=[5027, 4973]
Informative features: 0–89; noise: 90–499


## 3. Tree configuration and benchmark runner

The parameters below retain the configuration from the previous notebook. scikit-learn split importance is computed by counting how often each feature appears at internal nodes across the 40 trees, matching the meaning of Bankai's split-count importance.

In [11]:
N_JOBS = -1
N_WARMUPS = 1
N_REPEATS = 10
N_ESTIMATORS = 40
MAX_DEPTH = 20
MAX_LEAF_NODES = 511
MIN_SAMPLES_LEAF = 5
SAMPLE_FRACTION = 0.8
MODEL_SEED = 42
MAX_FEATURES = None
MAX_BINS = 63
BIN_SAMPLE_SIZE = 200_000

COMMON_FOREST_PARAMS = dict(
    n_estimators=N_ESTIMATORS, criterion="gini", max_depth=MAX_DEPTH,
    max_leaf_nodes=MAX_LEAF_NODES, min_samples_leaf=MIN_SAMPLES_LEAF,
    max_features=MAX_FEATURES, bootstrap=True, max_samples=SAMPLE_FRACTION,
    n_jobs=N_JOBS, random_state=MODEL_SEED,
)

def make_sklearn_rf(n_jobs=N_JOBS):
    return RandomForestClassifier(**{**COMMON_FOREST_PARAMS, "n_jobs": n_jobs})

def make_bankai(max_bins=None, importance_type="split"):
    params = dict(COMMON_FOREST_PARAMS)
    return BankaiRandomForestClassifier(
        **params, max_bins=max_bins, importance_type=importance_type,
        **({"bin_sample_size": BIN_SAMPLE_SIZE} if max_bins is not None else {}),
    )

def sklearn_split_importance(model):
    counts = np.zeros(N_FEATURES, dtype=np.float64)
    for tree in model.estimators_:
        used = tree.tree_.feature
        used = used[used >= 0]
        counts += np.bincount(used, minlength=N_FEATURES)
    total = counts.sum()
    return counts / total if total else counts

def run_benchmark(factories, importance_fns, native_importance=(), baseline=None):
    """Measure fit, importance, predict and quality; display per-run and median tables."""
    rows = []
    names = list(factories)
    native_importance = set(native_importance)
    for run in range(-N_WARMUPS, N_REPEATS):
        order = names if (run + N_WARMUPS) % 2 == 0 else list(reversed(names))
        for name in order:
            model = factories[name]()
            gc.collect()
            t0 = time.perf_counter(); model.fit(X_train, y_train); fit_s = time.perf_counter() - t0
            t0 = time.perf_counter(); importance = importance_fns[name](model); importance_s = time.perf_counter() - t0
            t0 = time.perf_counter(); predictions = model.predict(X_validation); predict_s = time.perf_counter() - t0
            # Probability inference and metric calculations are excluded from benchmark timings.
            probabilities = model.predict_proba(X_validation)
            positive_index = int(np.flatnonzero(model.classes_ == 1)[0])
            positive_scores = probabilities[:, positive_index]
            metrics = {
                "accuracy": accuracy_score(y_validation, predictions),
                "precision": precision_score(y_validation, predictions, zero_division=0),
                "recall": recall_score(y_validation, predictions, zero_division=0),
                "f1": f1_score(y_validation, predictions, zero_division=0),
                "roc_auc": roc_auc_score(y_validation, positive_scores),
            }
            assert len(importance) == N_FEATURES and len(predictions) == len(y_validation)
            if run >= 0:
                rows.append({"model": name, "run": run + 1, "fit_seconds": fit_s,
                             "importance_seconds": importance_s, "predict_seconds": predict_s,
                             "importance_in_fit": name in native_importance,
                             "fit_plus_importance_seconds": fit_s + (0 if name in native_importance else importance_s),
                             **metrics})
            del model, importance, predictions, probabilities, positive_scores
    samples = pd.DataFrame(rows)
    median_columns = ["fit_seconds", "importance_seconds", "predict_seconds",
                      "fit_plus_importance_seconds", "accuracy", "precision", "recall", "f1", "roc_auc"]
    medians = samples.groupby("model")[median_columns].median()
    if baseline:
        base = medians.loc[baseline]
        for metric in ("fit_seconds", "importance_seconds", "predict_seconds", "fit_plus_importance_seconds"):
            medians[metric.replace("_seconds", "_speedup_vs_baseline")] = base[metric] / medians[metric]
    print("Individual measurements (10 runs):")
    display(samples.drop(columns="importance_in_fit").round(5))
    print("Per-model medians:")
    display(medians.round(5))
    return samples, medians

## 4. Split importance: scikit-learn RF vs. exact Bankai

Bankai uses `max_bins=None`, so this scenario does not use histogram acceleration. Both implementations extract the per-feature split-node counts after fitting. scikit-learn importance is the normalized count derived from internal nodes; Bankai's `split` importance is the corresponding native importance.

In [12]:
split_factories = {
    "scikit-learn RF — split count": make_sklearn_rf,
    "Exact Bankai — split": lambda: make_bankai(max_bins=None, importance_type="split"),
}
split_importance_fns = {
    "scikit-learn RF — split count": sklearn_split_importance,
    "Exact Bankai — split": lambda model: model.feature_importances_,
}
split_samples, split_medians = run_benchmark(
    split_factories, split_importance_fns, baseline="scikit-learn RF — split count"
)

Individual measurements (10 runs):


,model,run,fit_seconds,importance_seconds,predict_seconds,fit_plus_importance_seconds,accuracy,precision,recall,f1,roc_auc
0,Exact Bankai — split,1,9.37727,0.00000,0.00857,9.37727,0.5615,0.57526,0.45327,0.50703,0.58779
1,scikit-learn RF — split count,1,12.50470,0.00066,0.01430,12.50537,0.5555,0.55321,0.55377,0.55349,0.57855
2,scikit-learn RF — split count,2,13.50976,0.00030,0.01314,13.51006,0.5555,0.55321,0.55377,0.55349,0.57855
3,Exact Bankai — split,2,11.17933,0.00000,0.00872,11.17933,0.5615,0.57526,0.45327,0.50703,0.58779
4,Exact Bankai — split,3,10.51026,0.00000,0.00675,10.51026,0.5615,0.57526,0.45327,0.50703,0.58779
5,scikit-learn RF — split count,3,13.78267,0.00036,0.01407,13.78303,0.5555,0.55321,0.55377,0.55349,0.57855
6,scikit-learn RF — split count,4,13.62586,0.00052,0.01193,13.62638,0.5555,0.55321,0.55377,0.55349,0.57855
7,Exact Bankai — split,4,10.65918,0.00000,0.00828,10.65918,0.5615,0.57526,0.45327,0.50703,0.58779
8,Exact Bankai — split,5,11.35230,0.00002,0.00965,11.35231,0.5615,0.57526,0.45327,0.50703,0.58779
9,scikit-learn RF — split count,5,14.19207,0.00043,0.01220,14.19251,0.5555,0.55321,0.55377,0.55349,0.57855


Per-model medians:


,fit_seconds,importance_seconds,predict_seconds,fit_plus_importance_seconds,accuracy,precision,recall,f1,roc_auc,fit_speedup_vs_baseline,importance_speedup_vs_baseline,predict_speedup_vs_baseline,fit_plus_importance_speedup_vs_baseline
model,,,,,,,,,,,,,
Exact Bankai — split,11.12239,0.00000,0.00867,11.12239,0.5615,0.57526,0.45327,0.50703,0.58779,1.24038,259.95134,1.61844,1.24041
scikit-learn RF — split count,13.79602,0.00042,0.01403,13.79635,0.5555,0.55321,0.55377,0.55349,0.57855,1.00000,1.00000,1.00000,1.00000


## 5. Permutation importance: scikit-learn vs. native Bankai

For scikit-learn, we measure validation accuracy permutation importance with five permutations per feature. To avoid nested parallelism, prediction within this step uses one thread and features are distributed across `n_jobs=-1`. In Bankai, native OOB importance is computed inside `fit`; reading `feature_importances_` after training only accesses the result. Compare `fit_plus_importance_seconds` for the total fit-plus-importance cost: Bankai already accounts for importance during fitting.

In [13]:
PERMUTATION_REPEATS = 5
SKLEARN_PERM_NAME = "scikit-learn RF — permutation (validation)"
BANKAI_PERM_NAME = "Exact Bankai — permutation OOB native"

def sklearn_permutation(model):
    model.set_params(n_jobs=1)
    result = permutation_importance(
        model, X_validation, y_validation, scoring="accuracy",
        n_repeats=PERMUTATION_REPEATS, random_state=MODEL_SEED, n_jobs=N_JOBS,
    )
    return result.importances_mean

permutation_factories = {
    SKLEARN_PERM_NAME: make_sklearn_rf,
    BANKAI_PERM_NAME: lambda: make_bankai(max_bins=None, importance_type="permutation"),
}
permutation_importance_fns = {
    SKLEARN_PERM_NAME: sklearn_permutation,
    BANKAI_PERM_NAME: lambda model: model.feature_importances_,
}
permutation_samples, permutation_medians = run_benchmark(
    permutation_factories, permutation_importance_fns,
    native_importance={BANKAI_PERM_NAME}, baseline=SKLEARN_PERM_NAME,
)

Individual measurements (10 runs):


,model,run,fit_seconds,importance_seconds,predict_seconds,fit_plus_importance_seconds,accuracy,precision,recall,f1,roc_auc
0,Exact Bankai — permutation OOB native,1,11.88003,0.00001,0.00952,11.88003,0.5615,0.57526,0.45327,0.50703,0.58779
1,scikit-learn RF — permutation (validation),1,14.49655,9.14671,0.01541,23.64326,0.5555,0.55321,0.55377,0.55349,0.57855
2,scikit-learn RF — permutation (validation),2,14.58741,9.16709,0.01567,23.75450,0.5555,0.55321,0.55377,0.55349,0.57855
3,Exact Bankai — permutation OOB native,2,11.79446,0.00000,0.01058,11.79446,0.5615,0.57526,0.45327,0.50703,0.58779
4,Exact Bankai — permutation OOB native,3,11.81156,0.00000,0.00991,11.81156,0.5615,0.57526,0.45327,0.50703,0.58779
5,scikit-learn RF — permutation (validation),3,14.32794,9.13191,0.01620,23.45985,0.5555,0.55321,0.55377,0.55349,0.57855
6,scikit-learn RF — permutation (validation),4,14.90567,9.11929,0.01560,24.02496,0.5555,0.55321,0.55377,0.55349,0.57855
7,Exact Bankai — permutation OOB native,4,11.75559,0.00000,0.01053,11.75559,0.5615,0.57526,0.45327,0.50703,0.58779
8,Exact Bankai — permutation OOB native,5,12.19139,0.00000,0.00844,12.19139,0.5615,0.57526,0.45327,0.50703,0.58779
9,scikit-learn RF — permutation (validation),5,14.15881,9.46869,0.01200,23.62750,0.5555,0.55321,0.55377,0.55349,0.57855


Per-model medians:


,fit_seconds,importance_seconds,predict_seconds,fit_plus_importance_seconds,accuracy,precision,recall,f1,roc_auc,fit_speedup_vs_baseline,importance_speedup_vs_baseline,predict_speedup_vs_baseline,fit_plus_importance_speedup_vs_baseline
model,,,,,,,,,,,,,
Exact Bankai — permutation OOB native,11.99685,0.00000,0.00945,11.99685,0.5615,0.57526,0.45327,0.50703,0.58779,1.21813,6.129856e+06,1.63162,1.99777
scikit-learn RF — permutation (validation),14.61367,9.44926,0.01542,23.96690,0.5555,0.55321,0.55377,0.55349,0.57855,1.00000,1.000000e+00,1.00000,1.00000


## 6. Histograms: Bankai vs. LightGBM RF

Both use 63-bin histograms and gain importance. The main tree parameters are retained; LightGBM RF uses sampling without replacement (`bagging_fraction=0.8`, `bagging_freq=1`), while Bankai uses bootstrap sampling. Keep this sampling difference and the algorithms' internal differences in mind when interpreting timings and importance values.

In [14]:
LIGHTGBM_RF_PARAMS = dict(
    boosting_type="rf", objective="binary", n_estimators=N_ESTIMATORS,
    max_bin=MAX_BINS, max_depth=MAX_DEPTH, num_leaves=MAX_LEAF_NODES,
    min_child_samples=MIN_SAMPLES_LEAF, bagging_freq=1,
    bagging_fraction=SAMPLE_FRACTION, feature_fraction=1.0,
    subsample_for_bin=BIN_SAMPLE_SIZE, n_jobs=N_JOBS,
    random_state=MODEL_SEED, importance_type="gain", verbosity=-1,
)

def make_lightgbm_rf():
    return LGBMClassifier(**LIGHTGBM_RF_PARAMS)

hist_factories = {
    "Bankai histograms (63 bins) — gain": lambda: make_bankai(max_bins=MAX_BINS, importance_type="gain"),
    "LightGBM RF (63 bins) — gain": make_lightgbm_rf,
}
hist_importance_fns = {
    "Bankai histograms (63 bins) — gain": lambda model: model.feature_importances_,
    "LightGBM RF (63 bins) — gain": lambda model: model.feature_importances_,
}
hist_samples, hist_medians = run_benchmark(
    hist_factories, hist_importance_fns, baseline="LightGBM RF (63 bins) — gain"
)

Individual measurements (10 runs):


,model,run,fit_seconds,importance_seconds,predict_seconds,fit_plus_importance_seconds,accuracy,precision,recall,f1,roc_auc
0,LightGBM RF (63 bins) — gain,1,4.69315,0.00006,0.00561,4.69321,0.570,0.56853,0.56281,0.56566,0.59361
1,Bankai histograms (63 bins) — gain,1,2.21542,0.00002,0.03430,2.21544,0.554,0.56212,0.46834,0.51096,0.57696
2,Bankai histograms (63 bins) — gain,2,1.99553,0.00000,0.02684,1.99553,0.554,0.56212,0.46834,0.51096,0.57696
3,LightGBM RF (63 bins) — gain,2,4.30007,0.00005,0.00416,4.30013,0.570,0.56853,0.56281,0.56566,0.59361
4,LightGBM RF (63 bins) — gain,3,4.16137,0.00005,0.00518,4.16142,0.570,0.56853,0.56281,0.56566,0.59361
5,Bankai histograms (63 bins) — gain,3,1.91437,0.00000,0.02698,1.91437,0.554,0.56212,0.46834,0.51096,0.57696
6,Bankai histograms (63 bins) — gain,4,1.82842,0.00000,0.02426,1.82842,0.554,0.56212,0.46834,0.51096,0.57696
7,LightGBM RF (63 bins) — gain,4,4.11099,0.00005,0.00408,4.11104,0.570,0.56853,0.56281,0.56566,0.59361
8,LightGBM RF (63 bins) — gain,5,4.20315,0.00005,0.00525,4.20321,0.570,0.56853,0.56281,0.56566,0.59361
9,Bankai histograms (63 bins) — gain,5,1.98451,0.00001,0.03055,1.98452,0.554,0.56212,0.46834,0.51096,0.57696


Per-model medians:


,fit_seconds,importance_seconds,predict_seconds,fit_plus_importance_seconds,accuracy,precision,recall,f1,roc_auc,fit_speedup_vs_baseline,importance_speedup_vs_baseline,predict_speedup_vs_baseline,fit_plus_importance_speedup_vs_baseline
model,,,,,,,,,,,,,
Bankai histograms (63 bins) — gain,1.90935,0.00000,0.02524,1.90935,0.554,0.56212,0.46834,0.51096,0.57696,2.25033,25.94775,0.19775,2.25035
LightGBM RF (63 bins) — gain,4.29667,0.00005,0.00499,4.29672,0.570,0.56853,0.56281,0.56566,0.59361,1.00000,1.00000,1.00000,1.00000


## Reading the tables

For each scenario, the first table contains the values from each measured run; the second contains per-model medians. Times are shown in seconds. The median table also shows speedup against the reference model for `fit`, importance, `predict`, and fit plus external importance.

`accuracy`, `precision`, `recall`, `f1`, and `roc_auc` are calculated on the validation split and are excluded from the timed measurements. `roc_auc` uses positive-class probabilities. Bankai's native OOB importance is computed during `fit`, so `fit_plus_importance_seconds` does not add the time to read the native result a second time.

The benchmark runs one warm-up and ten measured iterations. Results depend on the hardware and runtime shown at the beginning.